# Week 1 – Monthly Dataset Aggregation

**Objective:** Combine monthly listing and sold extracts into unified Residential-only datasets while protecting source lineage and validating every transformation.

This notebook is a backfilled explanation of the completed Week 1 pipeline. Reusable implementation remains in `src/week1_aggregate.py`.

## Business reason

Trend analysis requires a consistent multi-month dataset. The monthly files could not be concatenated blindly because their schemas changed over time, some listing headers contained repeated names, and historical sold files contained optional filled-coordinate fields.

## Reproducible project setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.week1_aggregate import (
    END_MONTH,
    OUTPUT_FILENAMES,
    START_MONTH,
    discover_and_inspect,
    month_range,
    property_type_summary,
)
from src.weeks2_3_validate import EXPECTED_ROWS, INPUT_FILES

print(f"Expected source coverage: {START_MONTH} through {END_MONTH}")
print(f"Expected months per dataset: {len(month_range(START_MONTH, END_MONTH))}")

Expected source coverage: 202401 through 202609
Expected months per dataset: 33


## Aggregation results

In [2]:
aggregation_summary = pd.DataFrame(
    {
        "dataset": ["listings", "sold"],
        "combined_rows_before_filter": [1_031_314, 736_548],
        "residential_rows_after_filter": [655_943, 495_334],
        "processed_columns": [75, 86],
        "source_months": [33, 33],
    }
)

aggregation_summary["rows_removed_by_filter"] = (
    aggregation_summary["combined_rows_before_filter"]
    - aggregation_summary["residential_rows_after_filter"]
)
aggregation_summary["residential_share_percent"] = (
    aggregation_summary["residential_rows_after_filter"]
    / aggregation_summary["combined_rows_before_filter"]
    * 100
).round(2)

aggregation_summary

,dataset,combined_rows_before_filter,residential_rows_after_filter,processed_columns,source_months,rows_removed_by_filter,residential_share_percent
0,listings,1031314,655943,75,33,375371,63.60
1,sold,736548,495334,86,33,241214,67.25


## Property types before filtering

This aggregate audit is run against the monthly source files and does not display MLS records. It documents the unique categories and Residential share required by the handbook.

In [3]:
audits, unrecognized_files = discover_and_inspect()
if unrecognized_files:
    raise ValueError(f"Unrecognized source files: {unrecognized_files}")

listing_property_types = property_type_summary(audits, "listings")
sold_property_types = property_type_summary(audits, "sold")

display(listing_property_types)
display(sold_property_types)

del audits

,property_type,record_count,share_percent
0,Residential,655943,63.602647
1,ResidentialLease,214277,20.777086
2,Land,66848,6.481828
3,ResidentialIncome,38287,3.712448
4,ManufacturedInPark,29386,2.849375
5,CommercialSale,14153,1.372327
6,CommercialLease,9207,0.892745
7,BusinessOpportunity,3213,0.311544


,property_type,record_count,share_percent
0,Residential,495334,67.250743
1,ResidentialLease,169471,23.008820
2,Land,23310,3.164763
3,ResidentialIncome,19797,2.687809
4,ManufacturedInPark,19771,2.684279
5,CommercialSale,4557,0.618697
6,CommercialLease,3833,0.520401
7,BusinessOpportunity,475,0.064490


### Interpretation

- Listings: 1,031,314 monthly records were combined; 655,943 Residential records (63.60%) were retained.
- Sold: 736,548 monthly records were combined; 495,334 Residential records (67.25%) were retained.
- Both processed datasets contain all 33 source months from January 2024 through September 2026.
- The counts describe combined extract records. They should not yet be interpreted as unique properties or unique transactions.

## Important data-quality discoveries

1. **Damaged source file:** The original January 2026 listing extract was truncated. The preflight correctly stopped before creating processed outputs. The month was regenerated and validated at 25,933 structurally complete records.
2. **Repeated listing headers:** Eleven listing column names appeared twice in historical source headers. Their values agreed in structurally complete records, and the pandas-created duplicate copies were removed intentionally.
3. **Schema evolution:** Three ordered listing schema groups and six ordered sold schema groups were identified.
4. **Filled coordinates:** Seven historical sold files contained populated `latfilled` and `lonfilled` fields. These were preserved rather than silently discarded.
5. **Property categories:** Eight pre-filter property types were found in each source family. Residential accounts for 63.60% of listing records and 67.25% of sold records.
6. **Lineage:** `_source_month` and `_source_file` were added so each combined record can be traced to its source extract.

## Safe output verification

This cell checks only private-file existence and expected metadata. It does not load or display rows.

In [4]:
processed_outputs = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "expected_rows": EXPECTED_ROWS[dataset],
            "output_filename": OUTPUT_FILENAMES[dataset],
            "file_exists_locally": INPUT_FILES[dataset].exists(),
        }
        for dataset in ("listings", "sold")
    ]
)

processed_outputs

,dataset,expected_rows,output_filename,file_exists_locally
0,listings,655943,combined_listings_residential.csv,True
1,sold,495334,combined_sold_residential.csv,True


## Design decisions

- Validate all source coverage and structure before concatenation.
- Stop without writing outputs when a blocking error is found.
- Filter both datasets with the exact rule `PropertyType == "Residential"`.
- Preserve optional fields through schema union.
- Process listings and sold separately to limit memory use.
- Store processed CSV files privately under `data/processed/`.

## Reproduction

The complete preflight and aggregation can be rerun from the repository root with:

```bash
.venv/bin/python src/week1_aggregate.py
```

This command rewrites the two private processed CSVs after all validations pass. It should be run deliberately, not as a routine notebook cell.

## Conclusion

Week 1 produced structurally validated, Residential-only inputs through September 2026. The property-type audit also documents the exact population removed by the `PropertyType == "Residential"` filter.

**Next chapter:** `02_03_data_validation_eda.ipynb`